In [8]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import urllib.request
import time

# ==========================================
# පියවර 1: AI Model එක ඩවුන්ලෝඩ් කිරීම
# ==========================================
model_path = 'blaze_face_short_range.tflite'

if not os.path.exists(model_path):
    print("AI Model eka download wenawa. Poddak inna...")
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, model_path)
    print("Download eka iwarai!\n")

# ==========================================
# පියවර 2: ෆෝල්ඩර් එක සහ AI එක Setup කිරීම
# ==========================================
user_name = input("enter your username: ")
dataset_folder = f"data_set/{user_name}"

if not os.path.exists(dataset_folder):
    os.makedirs(dataset_folder)
    print(f"created dataset folder: {dataset_folder}")
else:
    print(f"dataset folder already exists: {dataset_folder}")

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
detector = vision.FaceDetector.create_from_options(options)

count = 0 # ෆොටෝ ගණන් කරන්න පාවිච්චි කරන විචල්‍යය

# ==========================================
# පියවර 3: කැමරාව ඔන් කිරීම සහ මූණ සේව් කිරීම
# ==========================================
# මෙතන 0 හරි 1 හරි අංකය දීලා, අගට V4L2 කෑල්ල අනිවාර්යයෙන්ම දාන්න
# කැමරාව Initialize කිරීම
cap = cv2.VideoCapture(0)

print(f"\n{user_name} ge photos save wenna patan gannawa. Camerewa diha balan inna...")

while True:
    success, frame = cap.read()
    if not success:
        print("camera not available")
        break

    # කොළ පාට කොටුව අඳින්න කලින් ඔරිජිනල් ෆොටෝ එකේ කොපියක් ගන්නවා
    clean_frame = frame.copy()

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    detection_result = detector.detect(mp_image)

    if detection_result.detections:
        # අපි පළවෙනියට හම්බවෙන මූණ විතරක් ගන්නවා (කැමරාවේ එක්කෙනෙක් ඉන්නවා කියලා හිතලා)
        detection = detection_result.detections[0]

        bbox = detection.bounding_box
        x = int(bbox.origin_x)
        y = int(bbox.origin_y)
        w = int(bbox.width)
        h = int(bbox.height)

        # මූණ කැමරා රාමුවෙන් එලියට ගියොත් කෝඩ් එක Crash වෙන එක නවත්තන්න සීමාවන් හදනවා
        h_frame, w_frame, _ = frame.shape
        x, y = max(0, x), max(0, y)
        w, h = min(w, w_frame - x), min(h, h_frame - y)

        # ස්ක්‍රීන් එකේ පේන ෆොටෝ එකේ කොටුව අඳිනවා
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # අර අපි ගත්ත clean_frame (පිරිසිදු කොපියෙන්) මූණ විතරක් කපනවා
        face_crop = clean_frame[y:y+h, x:x+w]

        # කපපු ෆොටෝ එකේ ප්‍රමාණයක් තියෙනවද කියලා බලනවා (බිංදුවක් නෙවෙයි නම්)
        if face_crop.size != 0:
            # ෆොටෝ එක ෆෝල්ඩර් එකට සේව් කරනවා (උදා: chamindu_0.jpg, chamindu_1.jpg)
            file_path = f"{dataset_folder}/{user_name}_{count}.jpg"
            cv2.imwrite(file_path, face_crop)
            count += 1

            # ස්ක්‍රීන් එකේ ෆොටෝ කීයක් සේව් වුණාද කියලා පෙන්නනවා
            cv2.putText(frame, f"Saved: {count}/20", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # වෙනස් කෝණ (Angles) වලින් ෆොටෝ වැදෙන්න පොඩි වෙලාවක් දෙනවා
            time.sleep(0.5)

    cv2.imshow("Face Data Collector", frame)

    # ෆොටෝ 20ක් සේව් වුණාම ලූප් එකෙන් එලියට එනවා
    if count >= 20:
        print(f"\nNiyamai! {user_name} ge photos 20k save wela iwarai! ✅")
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


dataset folder already exists: data_set/chamindu

chamindu ge photos save wenna patan gannawa. Camerewa diha balan inna...


W0000 00:00:1782836786.756956   28623 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



Niyamai! chamindu ge photos 20k save wela iwarai! ✅


In [3]:
q